# Module 1 Homework: Agentic RAG

## LLM Provider Setup

This notebook uses **Groq** (free tier — no credit card needed).

Get a free key at https://console.groq.com → API Keys.
Then set it in your terminal before launching Jupyter:
```bash
export GROQ_API_KEY=gsk_...
```

Q1, Q2, Q4 run with zero API calls. Only Q3, Q5, Q6 need the key.

In [1]:
import os
from openai import OpenAI

# Groq is OpenAI-compatible — same SDK, different base_url
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
)

MODEL = "llama-3.3-70b-versatile"  # free on Groq

## Q1. How many lesson pages?

Load docs from GitHub (pinned to commit `8c1834d`).

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print(f"Q1 answer: {len(documents)} lesson pages")

Q1 answer: 72 lesson pages


## Q2. Indexing and searching

Index with minsearch, run a search, check first result.

In [3]:
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

q2_query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(q2_query)

print(f"Q2 answer: {results[0]['filename']}")
print("\nTop 5:")
for r in results[:5]:
    print(f"  {r['filename']}")

Q2 answer: 01-agentic-rag/lessons/14-agentic-loop.md

Top 5:
  01-agentic-rag/lessons/14-agentic-loop.md
  01-agentic-rag/lessons/15-frameworks.md
  01-agentic-rag/lessons/13-function-calling.md
  01-agentic-rag/lessons/11-agents-intro.md
  01-agentic-rag/lessons/16-other-frameworks.md


## Q3. RAG — count input tokens

**Requires GROQ_API_KEY**

In [4]:
import sys
sys.path.insert(0, '.')
from rag_helper import RAGBase

rag = RAGBase(index=index, llm_client=client, model=MODEL)

answer_q3, usage_q3 = rag.rag(q2_query)

print(f"Input tokens  : {usage_q3.prompt_tokens}")
print(f"Output tokens : {usage_q3.completion_tokens}")
print(f"\nAnswer:\n{answer_q3}")

Input tokens  : 7219
Output tokens : 19

Answer:
I'll answer your questions based on the provided context. Go ahead and ask your question.


## Q4. Chunking

No API needed — just count chunks.

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Q4 answer: {len(chunks)} chunks")

Q4 answer: 295 chunks


## Q5. RAG with chunking — compare input tokens

**Requires GROQ_API_KEY**

In [6]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

rag_chunked = RAGBase(index=chunk_index, llm_client=client, model=MODEL)

answer_q5, usage_q5 = rag_chunked.rag(q2_query)

ratio = usage_q3.prompt_tokens / usage_q5.prompt_tokens

print(f"Full-page tokens : {usage_q3.prompt_tokens}")
print(f"Chunked tokens   : {usage_q5.prompt_tokens}")
print(f"Ratio            : {ratio:.1f}x fewer with chunking")
print(f"\nQ5 answer: ~{round(ratio)}x fewer input tokens")

Full-page tokens : 7219
Chunked tokens   : 2346
Ratio            : 3.1x fewer with chunking

Q5 answer: ~3x fewer input tokens


## Q6. Agentic loop — count search tool calls

**Requires GROQ_API_KEY**

Note: Groq doesn't support the OpenAI Responses API (`responses.create`),
so we use the Chat Completions API + toyaikit's `OpenAIChatCompletionsClient`.

In [7]:
import json
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner
from toyaikit.chat.interface import IPythonChatInterface

def search(query: str) -> str:
    """Search the course lessons for content relevant to the query."""
    results = chunk_index.search(query, num_results=3)
    return "\n\n".join(r["content"] for r in results)

tools = Tools()
tools.add_tool(search)

llm_client = OpenAIChatCompletionsClient(model=MODEL, client=client)

runner = OpenAIChatCompletionsRunner(
    tools=tools,
    developer_prompt=(
        "You're a course teaching assistant. Answer the student's question using "
        "the search tool. Make multiple searches with different keywords before answering."
    ),
    chat_interface=IPythonChatInterface(),
    llm_client=llm_client,
)

q6_query = "How does the agentic loop work, and how is it different from plain RAG?"
messages = runner.loop(prompt=q6_query)

# Count how many times the model called the search tool
search_calls = 0
for m in messages:
    # Chat completions: tool calls are on the message object
    if hasattr(m, 'tool_calls') and m.tool_calls:
        search_calls += len(m.tool_calls)
    elif isinstance(m, dict) and m.get('role') == 'assistant':
        calls = m.get('tool_calls') or []
        search_calls += len(calls)

print(f"Q6 answer: search() called {search_calls} times")

/Users/dt00035/projects/personal/reference/llm-zoomcamp/cohorts/2026/01-agentic-rag/.venv/lib/python3.13/site-packages/toyaikit/chat/runners.py:542: UnknownModelWarning: No pricing data for model 'llama-3.3-70b-versatile'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(


TypeError: 'LoopResult' object is not iterable

## All answers

In [ ]:
print("=" * 55)
print("ANSWERS")
print("=" * 55)
print(f"Q1  lesson pages         = {len(documents)}")
print(f"Q2  first search result  = {results[0]['filename']}")
try:
    print(f"Q3  input tokens (full)  = {usage_q3.prompt_tokens}")
    print(f"Q4  chunk count          = {len(chunks)}")
    print(f"Q5  input tokens (chunk) = {usage_q5.prompt_tokens}  (~{round(ratio)}x fewer)")
    print(f"Q6  search calls         = {search_calls}")
except NameError:
    print("Q3/Q5/Q6: set GROQ_API_KEY and re-run those cells")
    print(f"Q4  chunk count          = {len(chunks)}")